# The days and hours that break the day-ahead ridge

The day-ahead ridge is the forecast to inspect. Penalty 1, day-ahead inputs only, no search. This notebook lists where that one forecast is wrong, in clock time and on particular dates.

A positive residual is an hour that was dirtier than the forecast. That is the miss that places work in a dirty hour and understates a carbon report. A negative residual is the opposite: the forecast avoided an hour that was actually fine.

Blocks of the day:

| Block | Local hours | Why it is separate |
| --- | --- | --- |
| Night | 0–5 | Gas is on, solar is zero, the level is high and flat. |
| Morning ramp | 6–9 | Solar arrives and carbon falls. |
| Solar window | 10–15 | The clean hours a shift wants. |
| Evening ramp | 16–20 | Solar leaves and gas returns. This is the fastest change of the day. |
| Late evening | 21–23 | Back toward the overnight level. |

In [1]:
%matplotlib inline

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12

from src.carbon_data import load_model_frame
from src.modeling import (
    TARGET,
    day_scores,
    error_components,
    fixed_forecasts,
    local_day_labels,
    train_test_split,
)

frame = load_model_frame()
train, test = train_test_split(frame)
assert train.index.max() < test.index.min()
forecasts = fixed_forecasts(train, test)
print(
    f"train through {train.index.max():%Y-%m-%d %H} UTC; "
    f"holdout {test.index.min():%Y-%m-%d %H} to {test.index.max():%Y-%m-%d %H} UTC "
    f"({len(test)} hours)"
)
print(
    f"mean carbon {train[TARGET].mean():.0f} g/kWh in training, "
    f"{test[TARGET].mean():.0f} on the holdout"
)
print("forecasts:", ", ".join(forecasts))


/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in ma

train through 2026-09-20 22 UTC; holdout 2026-09-20 23 to 2026-10-18 23 UTC (673 hours)
mean carbon 103 g/kWh in training, 172 on the holdout
forecasts: hour-of-day mean, yesterday, same hour, day-ahead ridge, day-ahead tree, previous hour


/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/kareha/Desktop/EcoShift/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in ma

In [2]:
scored = test[["hour_local", "dow", "carbon_lag_24", TARGET]].copy()
scored["predicted"] = forecasts["day-ahead ridge"]
scored["too_clean"] = scored[TARGET] - scored["predicted"]
scored["local_day"] = local_day_labels(test).to_numpy()
scored["block"] = pd.cut(
    scored["hour_local"],
    bins=[-0.5, 5.5, 9.5, 15.5, 20.5, 23.5],
    labels=["night", "morning ramp", "solar window", "evening ramp", "late evening"],
)
blocks = scored.groupby("block", observed=True).agg(
    mae=("too_clean", lambda values: values.abs().mean()),
    bias_too_clean=("too_clean", "mean"),
    hours=("too_clean", "size"),
)
print("Day-ahead ridge by block of the day")
blocks.round(2)


Day-ahead ridge by block of the day


,mae,bias_too_clean,hours
block,,,
night,25.99,13.00,168
morning ramp,26.72,12.51,112
solar window,22.85,-4.64,168
evening ramp,25.31,2.56,141
late evening,25.00,11.15,84


In [3]:
daily = day_scores(test, forecasts["day-ahead ridge"])
yesterday = scored.groupby("local_day")["carbon_lag_24"].mean().rename("yesterday_mean")
daily = daily.merge(yesterday, left_on="local_day", right_index=True, how="left")
daily["level_jump"] = daily["actual_mean"] - daily["yesterday_mean"]
jump = daily["level_jump"].abs()
cut = jump.median()
jumped = daily[jump >= cut]
steady = daily[jump < cut]
print(
    f"Days the carbon level moved at least {cut:.0f} g/kWh from yesterday: "
    f"MAE {jumped['mae'].mean():.1f}, 4-hour regret {jumped['regret_4h'].mean():.1f}."
)
print(
    f"Days it moved less than that: "
    f"MAE {steady['mae'].mean():.1f}, 4-hour regret {steady['regret_4h'].mean():.1f}."
)
weekday = scored[scored["dow"] < 5]
weekend = scored[scored["dow"] >= 5]
print(
    f"Weekday MAE {weekday['too_clean'].abs().mean():.1f} g/kWh, "
    f"weekend MAE {weekend['too_clean'].abs().mean():.1f}."
)
worst = scored.reindex(scored["too_clean"].abs().sort_values(ascending=False).index).head(8)
worst = worst.assign(
    when=lambda frame: frame.index.tz_convert("America/Los_Angeles").strftime("%Y-%m-%d %H:%M")
)
worst[["when", "hour_local", TARGET, "predicted", "carbon_lag_24", "too_clean"]].round(1)


Days the carbon level moved at least 15 g/kWh from yesterday: MAE 32.0, 4-hour regret 2.0.
Days it moved less than that: MAE 17.2, 4-hour regret 2.0.
Weekday MAE 23.5 g/kWh, weekend MAE 29.0.


,when,hour_local,carbon_intensity_g_per_kwh,predicted,carbon_lag_24,too_clean
2026-10-17 16:00:00+00:00,2026-10-17 09:00,9,236.0,99.4,115.2,136.7
2026-10-17 15:00:00+00:00,2026-10-17 08:00,8,272.4,162.9,194.6,109.5
2026-10-17 17:00:00+00:00,2026-10-17 10:00,10,162.7,77.6,88.1,85.1
2026-10-17 13:00:00+00:00,2026-10-17 06:00,6,282.9,198.2,223.8,84.7
2026-10-17 14:00:00+00:00,2026-10-17 07:00,7,279.7,197.1,225.1,82.6
2026-10-17 12:00:00+00:00,2026-10-17 05:00,5,292.0,213.1,241.7,78.9
2026-10-17 18:00:00+00:00,2026-10-17 11:00,11,146.5,70.5,79.5,75.9
2026-10-17 11:00:00+00:00,2026-10-17 04:00,4,293.6,218.6,250.1,75.1


`too_clean` is actual minus predicted. The largest rows are the hours a shift or a report would get most wrong. `carbon_lag_24` is what yesterday's same hour looked like, which is the input the linear forecast leans on.